In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim import AdamW
import random

random.seed(42)
torch.manual_seed(42)

transform = transforms.Compose([
    transforms.ToTensor(),                                        # Pretvara sliku u PyTorch tenzor (vrednosti 0.0 - 1.0)
    transforms.Lambda(lambda img: torch.rot90(img, 1, [1, 2])), # Rotacija da slova ne budu "na bok"
    transforms.Lambda(lambda img: torch.flip(img, [2])),          # Horizontalno ogledalo ako je potrebno
    transforms.Normalize((0.5,), (0.5,))                          # Normalizacija na opseg [-1, 1]
])

train_dataset = datasets.EMNIST(
    root='./data', 
    split='byclass', 
    train=True, 
    download=True, 
    transform=transform
)
test_dataset = datasets.EMNIST(
    root='./data',
    split='byclass',
    train= False,
    download=True,
    transform=transform   
)

BATCH_SIZE=256

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True, pin_memory=True)
test_loader=DataLoader(test_dataset, batch_size=BATCH_SIZE,shuffle=True, pin_memory=True)

data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"Dimenzije batch-a slika: {images.shape}")  # Očekujemo [64, 1, 28, 28]
print(f"Dimenzije batch-a labela: {labels.shape}") # Očekujemo [64]

Dimenzije batch-a slika: torch.Size([256, 1, 28, 28])
Dimenzije batch-a labela: torch.Size([256])


In [2]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channles=1, patch_size=4, emb_dim=64, img_size=28): #64 filtera, in_ch je dimenzija boje
        super().__init__()
        self.patch_size = patch_size

        self.proj = nn.Conv2d( #izmerimo koliko se slazu filteri sa slikom
            in_channels=in_channles,
            out_channels=emb_dim,
            kernel_size=patch_size,
            stride=patch_size
        )
    def forward(self,x):
        x=self.proj(x)
        x=x.flatten(2) # pretvorimo u 'string', cuvamo prve dve dim
        x=x.transpose(1,2) # menjamo mesta 2 i 3 dim, da bi svaki od 49 patcheva dobio vektor od 64 dim// takav nam oblik treba za transformer

        return x


In [3]:
class ViTInputEmbedding(nn.Module):
    def __init__(self, in_channels=1,patch_size=4,emb_dim=64, img_size=28, num_classes=62): #num classes je broj razlicitih slova i brojeva, trebace kasnije
        super().__init__()
        self.patch_embed=PatchEmbedding(in_channels, patch_size, emb_dim, img_size)

        self.cls_token=nn.Parameter(torch.randn(1,1, emb_dim)) #nn.paremeter kaze torchu da se na njemu uci

        num_patches= (img_size//patch_size)**2
        self.pos_embedding= nn.Parameter(torch.randn(1, num_patches + 1, emb_dim))

    def forward(self,x):

        batch_size=x.shape[0]

        x=self.patch_embed(x)

        cls_tokens=self.cls_token.expand(batch_size,-1,-1)

        x=torch.cat((cls_tokens, x), dim=1)

        x=x+ self.pos_embedding # sabiranje pozicionih informacija je backpropagatabilno
        return x




In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim=64, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1=nn.LayerNorm(emb_dim)

        self.attn= nn.MultiheadAttention(embed_dim=emb_dim, num_heads=num_heads, dropout=dropout, batch_first=True)

        self.norm2 = nn.LayerNorm(emb_dim)

        hidden_dim= int(emb_dim * mlp_ratio)

        self.mlp= nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,emb_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):

        normed_x= self.norm1(x)

        attn_out, _= self.attn(normed_x, normed_x, normed_x)

        x=x+attn_out

        x=x+ self.mlp(self.norm2(x))

        return x



In [5]:
class VisionTransformer(nn.Module):
    def __init__(self, in_channels=1, patch_size=4, emb_dim=64, img_size=28, num_classes=62, depth=4, num_heads=4, dropout=0.1):
        super().__init__()
        self.embedding=ViTInputEmbedding(in_channels=in_channels, patch_size=patch_size, emb_dim=emb_dim, img_size=img_size, num_classes=num_classes)

        self.blocks= nn.ModuleList([
            TransformerBlock(emb_dim=emb_dim, num_heads=num_heads, dropout=dropout)
            for _ in range(depth)
        ])

        self.norm= nn.LayerNorm(emb_dim)

        self.head= nn.Linear(emb_dim, num_classes)

    def forward(self,x):
        x= self.embedding(x)

        for block in self.blocks:
            x= block(x)

        x=self.norm(x)

        cls_token_final = x[:,0]

        logits= self.head(cls_token_final)

        return logits


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Koristimo urejdaj: {device}")

model = VisionTransformer(
    in_channels=1,
    patch_size=4,
    emb_dim=64,
    img_size=28, 
    num_classes=62,
    depth=4,
    num_heads=4,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

print(f"Model je kreiran i prebacen na {device}")

Koristimo urejdaj: cuda
Model je kreiran i prebacen na cuda


In [57]:
num_epochs = 10
model.train()

for epoch in range(num_epochs):
    total_loss= 0.0
    correct=0
    total=0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss= criterion(logits, labels)

        loss.backward()

        #print("Da li je tenzor na GPU-u?", images.is_cuda)
        #print(f"Trenutna VRAM memorija u upotrebi: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        #print(f"Maksimalna VRAM memorija iskorišćena do sada: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")
        
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(logits.detach(), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


        if batch_idx % 500 == 0:
            print(f"Epoha [{epoch+1}/{num_epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    epoch_loss= total_loss/ len(train_loader)
    epoch_acc= 100*correct / total
    print(f"==> Kraj Epohe {epoch+1} | Prosečan Loss: {epoch_loss:.4f} | Tačnost (Accuracy): {epoch_acc:.2f}%\n")

Epoha [1/10] | Batch [0/2727] | Loss: 4.3564
Epoha [1/10] | Batch [500/2727] | Loss: 1.3176
Epoha [1/10] | Batch [1000/2727] | Loss: 0.8744
Epoha [1/10] | Batch [1500/2727] | Loss: 0.6578
Epoha [1/10] | Batch [2000/2727] | Loss: 0.6438
Epoha [1/10] | Batch [2500/2727] | Loss: 0.6170
==> Kraj Epohe 1 | Prosečan Loss: 1.0490 | Tačnost (Accuracy): 69.80%

Epoha [2/10] | Batch [0/2727] | Loss: 0.6381
Epoha [2/10] | Batch [500/2727] | Loss: 0.5664
Epoha [2/10] | Batch [1000/2727] | Loss: 0.6137
Epoha [2/10] | Batch [1500/2727] | Loss: 0.4980
Epoha [2/10] | Batch [2000/2727] | Loss: 0.6180
Epoha [2/10] | Batch [2500/2727] | Loss: 0.4674
==> Kraj Epohe 2 | Prosečan Loss: 0.5712 | Tačnost (Accuracy): 80.70%

Epoha [3/10] | Batch [0/2727] | Loss: 0.5516
Epoha [3/10] | Batch [500/2727] | Loss: 0.4490
Epoha [3/10] | Batch [1000/2727] | Loss: 0.4941
Epoha [3/10] | Batch [1500/2727] | Loss: 0.5326
Epoha [3/10] | Batch [2000/2727] | Loss: 0.5282
Epoha [3/10] | Batch [2500/2727] | Loss: 0.5099
==> Kr

In [58]:
torch.save({
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'vit_emnist.pth')

In [6]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionTransformer(
    in_channels=1,
    patch_size=4,
    emb_dim=64,
    img_size=28, 
    num_classes=62,
    depth=4,
    num_heads=4,
    dropout=0.1
).to(device)


optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)


checkpoint = torch.load('vit_emnist.pt', map_location=device)


model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print("Model je uspešno učitan sa diska!")

C:\Users\Nikola\AppData\Local\Temp\ipykernel_51784\3488416105.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('vit_emnist.pt', map_location=devi

Model je uspešno učitan sa diska!


In [7]:
model.eval()

VisionTransformer(
  (embedding): ViTInputEmbedding(
    (patch_embed): PatchEmbedding(
      (proj): Conv2d(1, 64, kernel_size=(4, 4), stride=(4, 4))
    )
  )
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=256, out_features=64, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=62, bias=True)
)

In [8]:
correct = 0
total = 0

with torch.no_grad():  # bez gradijenata
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Tačnost modela na test skupu: {test_accuracy:.2f}%")

Tačnost modela na test skupu: 86.31%


In [55]:
import tkinter as tk
from PIL import Image, ImageDraw, ImageOps
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output
from scipy.ndimage import center_of_mass

root = tk.Tk()
root.title("Vit EMINST test")

canvas_size=280
canvas= tk.Canvas(root, width=canvas_size, height=canvas_size, bg="black")
canvas.pack(pady=10)

app_image = Image.new("L", (canvas_size, canvas_size), color=0)
app_draw = ImageDraw.Draw(app_image)


In [56]:

def get_emnist_char(label):
    if label < 10:
        return str(label)
    elif label < 36:
        return chr(ord('A') + (label - 10))
    else:
        return chr(ord('a') + (label - 36))

def paint(event):
    x, y = event.x, event.y
    r = 15
  
    canvas.create_oval(x - r, y - r, x + r, y + r, fill="white", outline="white")
    app_draw.ellipse([x - r, y - r, x + r, y + r], fill=255)

canvas.bind("<B1-Motion>", paint)

label_result = tk.Label(root, text="Nacrtaj nešto i klikni Prepoznaj...", font=("Arial", 16, "bold"))
label_result.pack(pady=5)

def clear_canvas():
   
    canvas.delete("all")
    
    
    global app_image, app_draw
    app_image = Image.new("L", (canvas_size, canvas_size), color=0)
    app_draw = ImageDraw.Draw(app_image)
    
    
    label_result.config(text="Nacrtaj nešto i klikni Prepoznaj...")
    
btn_predict = tk.Button(root, text="Prepoznaj slovo", command=predict_drawing, font=("Arial", 12), bg="lightgreen")
btn_predict.pack(side=tk.LEFT, padx=20, pady=10)
btn_clear = tk.Button(root, text="Obriši platno", command=clear_canvas, font=("Arial", 12))
btn_clear.pack(pady=10)

root.mainloop()

--- MONTE CARLO DROPOUT (20 PROLAZA) ---
1. Klasa indeks 3 ('3'): Srednja verovatnoća = 100.0% (Nesigurnost/Std = ±0.0%)
2. Klasa indeks 61 ('z'): Srednja verovatnoća = 0.0% (Nesigurnost/Std = ±0.0%)
3. Klasa indeks 35 ('Z'): Srednja verovatnoća = 0.0% (Nesigurnost/Std = ±0.0%)
4. Klasa indeks 11 ('B'): Srednja verovatnoća = 0.0% (Nesigurnost/Std = ±0.0%)
5. Klasa indeks 5 ('5'): Srednja verovatnoća = 0.0% (Nesigurnost/Std = ±0.0%)

>>> REZULTAT: Prihvaćeno slovo '3'
--- MONTE CARLO DROPOUT (20 PROLAZA) ---
1. Klasa indeks 35 ('Z'): Srednja verovatnoća = 60.4% (Nesigurnost/Std = ±10.4%)
2. Klasa indeks 2 ('2'): Srednja verovatnoća = 22.8% (Nesigurnost/Std = ±9.0%)
3. Klasa indeks 61 ('z'): Srednja verovatnoća = 16.6% (Nesigurnost/Std = ±3.6%)
4. Klasa indeks 3 ('3'): Srednja verovatnoća = 0.1% (Nesigurnost/Std = ±0.1%)
5. Klasa indeks 27 ('R'): Srednja verovatnoća = 0.1% (Nesigurnost/Std = ±0.1%)

>>> REZULTAT: Prihvaćeno slovo 'Z'
--- MONTE CARLO DROPOUT (20 PROLAZA) ---
1. Klasa inde

In [ ]:
def predict_drawing():
    
    #plt.clf()
    #plt.close('all')
    img_resized = app_image.resize((28,28), Image.Resampling.LANCZOS)
    #img_resized = img_resized.point(lambda p: 255 if p > 50 else 0, mode='1').convert('L')
    img_resized= img_resized.rotate(270, expand=True)
    img_resized=ImageOps.mirror(img_resized)

    

    img_np = np.array(img_resized, dtype=np.float32) / 255.0

    if img_np.sum() > 0:
        cy, cx = center_of_mass(img_np)
        rows, cols = img_np.shape
        shift_y = int(rows / 2.0 - cy)
        shift_x = int(cols / 2.0 - cx)
        img_np = np.roll(img_np, shift_y, axis=0)
        img_np = np.roll(img_np, shift_x, axis=1)
    np.set_printoptions(linewidth=np.inf, precision=1, suppress=True)
    #print(img_np)
    # plt.figure(figsize=(3, 3))
    # plt.imshow(img_resized, cmap='gray')
    # plt.title("Ovo vidi model (28x28)")
    # plt.axis('off')
    # plt.show()
    
    #img_np = 1.0 - img_np
    #print(f"Maksimalna vrednost piksela na slici: {img_np.max()} ")
    tensor_img = torch.tensor(img_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tensor_img = torch.rot90(tensor_img, 1, [2, 3])
    tensor_img = torch.flip(tensor_img, [3])
    # Normalizacija na [-1, 1]
    tensor_img = (tensor_img - 0.5) / 0.5
    
    model.eval()

    with torch.no_grad():
        # outputs= model(tensor_img)
        # #print("Raw logits:", outputs[0][:10])
        # probs = F.softmax(outputs, dim=1)
        # #print("Raw probs:", probs[0][:])
        # confidance, predicted = torch.max(probs,1)
        # print("predicted.item:", predicted.item())
        outputs = model(tensor_img)
        probabilities = F.softmax(outputs, dim=1)
        
        # Uzimamo TOP 5 predviđanja umesto samo jednog
        top5_prob, top5_indices = torch.topk(probabilities, 5, dim=1)

    #print("--- TOP 5 PREDIKCIJA ---")
    #for i in range(5):
        #idx = top5_indices[0][i].item()
        #prob = top5_prob[0][i].item() * 100
        #char = get_emnist_char(idx)
        #print(f"{i+1}. Klasa indeks {idx} ('{char}'): {prob:.1f}%")



    pred_idx= top5_indices[0][0].item()
    conf_val= top5_prob[0][0].item()*100

    char_result = get_emnist_char(pred_idx)
    #print(f"Predviđeni indeks: {pred_idx}, Karakter: {char_result}")
    
    label_result.config(text=f"Predikcija: '{char_result}' ({conf_val:.1f}% pouzdanost)")


 

In [ ]:
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np


emnist_train = datasets.EMNIST(
    root='./data', 
    split='byclass', 
    train=True, 
    download=True
)


sample_idx = 1011
img_pil, label_idx = emnist_train[sample_idx]


img_np = np.array(img_pil, dtype=np.float32) / 255.0


np.set_printoptions(precision=1, suppress=True, linewidth=120)
print(f"--- NUMPY MATRICA PRAVOG EMNIST PRIMERA (Indeks klase: {label_idx}) ---")
print(img_np)


plt.figure(figsize=(3, 3))
plt.imshow(img_np, cmap='gray')


char_name = get_emnist_char(label_idx)
plt.title(f"Pravi EMNIST -> Klasa {label_idx} ('{char_name}')")
plt.axis('off')
plt.show()

In [54]:
def predict_drawing():
    
    img_resized = app_image.resize((28,28), Image.Resampling.LANCZOS)
    img_resized= img_resized.rotate(270, expand=True)
    img_resized=ImageOps.mirror(img_resized)

    

    img_np = np.array(img_resized, dtype=np.float32) / 255.0

    if img_np.sum() > 0:
        cy, cx = center_of_mass(img_np)
        rows, cols = img_np.shape
        shift_y = int(rows / 2.0 - cy)
        shift_x = int(cols / 2.0 - cx)
        img_np = np.roll(img_np, shift_y, axis=0)
        img_np = np.roll(img_np, shift_x, axis=1)
    np.set_printoptions(linewidth=np.inf, precision=1, suppress=True)
    
    
    tensor_img = torch.tensor(img_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tensor_img = torch.rot90(tensor_img, 1, [2, 3])
    tensor_img = torch.flip(tensor_img, [3])
    tensor_img = (tensor_img - 0.5) / 0.5
    
    # ovde divergira
    
    model.train()
    n_passes= 20
    all_probs=[]


    with torch.no_grad():
        for _ in range(n_passes):

            outputs = model(tensor_img)
            #print(f"outputs: {outputs.shape}")
            #print(outputs)
            probs = F.softmax(outputs, dim=1)
            #print(f"probs: {probs.shape}")
            all_probs.append(probs.cpu().numpy())
            #print(f"allprobs: {len(all_probs)}")
            #print(all_probs)


    all_probs =np.vstack(all_probs)
    #print(f"allprobs: {len(all_probs)}")
    #print(all_probs)

    mean_probs = np.mean(all_probs, 0)
    #print(f"mean_probs: {len(mean_probs)}")
    uncertainty= np.std(all_probs, 0)

    top5_ind = np.argsort(mean_probs)[::-1][:5]
    print(f"--- MONTE CARLO DROPOUT ({n_passes} PROLAZA) ---")
    for i, idx in enumerate(top5_ind):
        prob = mean_probs[idx] * 100
        unc = uncertainty[idx] * 100
        char = get_emnist_char(idx)
        print(f"{i+1}. Klasa indeks {idx} ('{char}'): Srednja verovatnoća = {prob:.1f}% (Nesigurnost/Std = ±{unc:.1f}%)")
    
    best_idx = top5_ind[0]
    best_conf = mean_probs[best_idx] * 100
    best_unc = uncertainty[best_idx] * 100
    best_char = get_emnist_char(best_idx)


    CONFIDANCE_THRESHOLD = 50.0
    UNCERTAINTY_THRESHOLD= 15.0

    if best_conf<CONFIDANCE_THRESHOLD or best_unc>UNCERTAINTY_THRESHOLD:
        label_result.config(text=f"Nepoznato / Nesigurno (Conf: {best_conf:.1f}%, Std: ±{best_unc:.1f}%)")
        print("\n>>> REZULTAT: Model odbacio unos jer nije dovoljno siguran!")
    else:
        label_result.config(text=f"Predikcija: '{best_char}' ({best_conf:.1f}%, ±{best_unc:.1f}%)")
        print(f"\n>>> REZULTAT: Prihvaćeno slovo '{best_char}'")


 